<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/4_model_training%20/4_1_model_random_forest_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4_1_model_random_forest

## Introducción y Resumen

El objetivo de esta notebook es generar las ventanas de entrada (X) y targets (y) para los horizontes de 30, 60 y 90 minutos, aplica el escalado de features sobre cada conjunto (train, valid, test) y finalmente guarda las ventanas escaladas en disco, dejándolas listas para el entrenamiento de modelos.

0. Configuración del Entorno

    Se conecta Google Drive y se clona el repositorio de trabajo. Se instalan e importan librerías necesarias como pandas, numpy, matplotlib y seaborn. Se cargan los datasets procesados previamente y se muestra un resumen de la información de los datasets.

1. Carga de datos

    Se importan los datasets procesados `mnq_train`, `mnq_valid` y `mnq_test`. Se revisa su estructura (filas, columnas, tipos de datos) y también de importa el listado de features seleccionados para cada ventana de tiempo.





## 0. Configuración del Entorno


### 0.1. Clonado de repositorio / Acceso a Drive

In [1]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit
!git clone https://github.com/GUNAPILLCO/neural_profit.git

fatal: destination path 'neural_profit' already exists and is not an empty directory.


In [2]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 0.2. Instalación de librerías


In [3]:
#!{sys.executable} -m pip install -q ta
#print("Librería instalada: technical-analysis")

In [4]:
!pip uninstall -y scikit-learn scikit-image threadpoolctl joblib --quiet
!pip install --no-cache-dir "scikit-learn==1.5.1" "optuna==3.6.1" "numpy>=1.24,<3" "scipy>=1.10" --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.4/308.4 kB 302.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.9.post2 requires scikit-learn>=1.6, but you have scikit-learn 1.5.1 which is incompatible.


In [5]:
!pip install optuna --quiet

### 0.3. Importación de librerías


In [6]:
# ==============================
# Librerías estándar de Python
# ==============================
import os
import sys
import re
import glob
import warnings
import requests
from datetime import datetime, timedelta
from functools import reduce

# ==============================
# Manejo y procesamiento de datos
# ==============================
import pandas as pd
import numpy as np
from tabulate import tabulate

# ==============================
# Visualización
# ==============================
import matplotlib.pyplot as plt

# ==============================
# Estadística
# ==============================
from scipy.stats import spearmanr

# ==============================
# Machine Learning y utilidades
# ==============================
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import sklearn, scipy
import sklearn, numpy, scipy, optuna
from sklearn.ensemble import RandomForestRegressor

import joblib
import optuna
from tqdm import tqdm

# ==============================
# Configuración general
# ==============================
warnings.filterwarnings("ignore")

import time

In [7]:
print("python:", sys.version)
print("sklearn:", sklearn.__version__)
print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)
print("optuna:", optuna.__version__)

python: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
sklearn: 1.5.1
numpy: 2.0.2
scipy: 1.16.2
optuna: 3.6.1


## 1. Carga de datos

### 1.1. Carga de datasets `mnq_train`, `mnq_valid` y `mnq_test`






In [8]:
def load_data(data: str):

    data_path = f'{drive_path}/3_dataset_preparation/mnq_{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [9]:
mnq_train = load_data("train")
mnq_valid = load_data("valid")
mnq_test = load_data("test")

### 1.2. Información de datasets


In [10]:
def info_dataset(df, name: str):
  print(f"Información del dataset {name}:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}\n")

  return num_dias, promedio_por_fecha

In [11]:
info_dataset(mnq_train, 'mnq_train')
info_dataset(mnq_valid, 'mnq_valid')
info_dataset(mnq_test, 'mnq_test')

Información del dataset mnq_train:

	Cantidad de días: 917
	Registros por día: 301
	Hora diaria de inicio 09:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York

Información del dataset mnq_valid:

	Cantidad de días: 197
	Registros por día: 301
	Hora diaria de inicio 09:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York

Información del dataset mnq_test:

	Cantidad de días: 197
	Registros por día: 301
	Hora diaria de inicio 09:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York



(197, np.float64(301.0))

### 1.3. Carga de listado de features por ventana de tiempo

In [12]:
import json

# Ruta al archivo guardado
path = f'{drive_path}/2_feature_engineering/features_list.json'

with open(path, "r") as f:
    features_dict = json.load(f)

# Extraer las listas
features_to_30 = features_dict["features_to_30"]
features_to_60 = features_dict["features_to_60"]
features_to_90 = features_dict["features_to_90"]


In [13]:
print(f'Listado de features para 30min: {features_to_30}')
print(f'Listado de features para 60min: {features_to_60}')
print(f'Listado de features para 90min: {features_to_90}')

Listado de features para 30min: ['ire_90', 'rev_mom_z_90', 'roc_60', 'rev_score_90', 'price_ema30']
Listado de features para 60min: ['ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'rev_mom_vol_z_60', 'momentum_5', 'roc_20']
Listado de features para 90min: ['ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'momentum_5', 'roc_20', 'rev_mom_vol_z_60']


## 2. Carga de ventanas `X_train_*_scaled`, `X_valid_*_scaled`, `X_test_*_scaled`

### 2.0. Funciones

#### Función para cargar ventanas

In [14]:
def load_windows_and_scaler(target: str, scaled=True):
    """
    Carga datasets (X, y) para train, valid y test junto con el scaler global.

    Parámetros
    ----------
    drive_path : str
        Ruta base donde se encuentran los archivos.
    scaled : bool, default=True
        Si True busca en la carpeta 'ventanas_x_y_scaled',
        si False en 'ventanas_x_y'.

    Retorna
    -------
    X_train, y_train, X_valid, y_valid, X_test, y_test, scaler
    """

    #Ruta de ventandas escaladas
    path_train  = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_train_{target}_scaled.npz'
    path_valid  = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_valid_{target}_scaled.npz'
    path_test   = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_test_{target}_scaled.npz'

    #Ruta de escalador
    path_scaler = f"{drive_path}/3_dataset_preparation/global_scaler_{target}.pkl"

    # Cargar npz
    data_train = np.load(path_train)
    data_valid = np.load(path_valid)
    data_test  = np.load(path_test)

    # Extraer X, y
    X_train, y_train = data_train["X"], data_train["y"]
    X_valid, y_valid = data_valid["X"], data_valid["y"]
    X_test,  y_test  = data_test["X"],  data_test["y"]

    # Cargar scaler
    scaler = joblib.load(path_scaler)

    return X_train, y_train, X_valid, y_valid, X_test, y_test, scaler


#### Función para revisar información de ventanas

In [15]:
def xy_info(target: str, X_train, y_train, X_valid, y_valid, X_test, y_test):
    print(f'Información para horizonte de {target} minutos:')

    for name, X, y in [
        ("entrenamiento", X_train, y_train),
        ("validación", X_valid, y_valid),
        ("testeo", X_test, y_test),
    ]:
        print(f'\nSet de {name}:')
        print(f'\t{X.shape[0]} ventanas (n_samples).')

        if X.ndim == 2:

            print(f'\t{X.shape[1]} features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_{target})')
        elif X.ndim == 3:

            print(f'\t{X.shape[1]} pasos en lookback × {X.shape[2]} features por paso. Dimensión 3D: (n_samples, window_size, len(features_{target})')

        print(f'\t{y.shape[0]} targets.')
        print(f'\tDistribución y: mean={y.mean():.6f}, std={y.std():.6f}, min={y.min():.6f}, max={y.max():.6f}')

    return  X_train.shape[0], X_valid.shape[0], X_test.shape[0]

### 2.1 Carga de ventanas 30 minutos

In [16]:
X_train_30_scaled, y_train_30, X_valid_30_scaled, y_valid_30, X_test_30_scaled, y_test_30, scaler_30 = load_windows_and_scaler(target = '30')

In [17]:
n_samples_train_30, n_samples_valid_30, n_samples_test_30 = xy_info( '30', X_train_30_scaled, y_train_30, X_valid_30_scaled, y_valid_30, X_test_30_scaled, y_test_30)

Información para horizonte de 30 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	193487 targets.
	Distribución y: mean=0.000056, std=0.002760, min=-0.028931, max=0.032372

Set de validación:
	41567 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	41567 targets.
	Distribución y: mean=0.000103, std=0.003273, min=-0.023167, max=0.068056

Set de testeo:
	41567 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	41567 targets.
	Distribución y: mean=-0.000023, std=0.002407, min=-0.015982, max=0.016097


### 2.2 Carga de ventanas 60 minutos

In [18]:
X_train_60_scaled, y_train_60, X_valid_60_scaled, y_valid_60, X_test_60_scaled, y_test_60, scaler_60 = load_windows_and_scaler(target = '60')

In [19]:
n_samples_train_60, n_samples_valid_60, n_samples_test_60 = xy_info( '60', X_train_60_scaled, y_train_60, X_valid_60_scaled, y_valid_60, X_test_60_scaled, y_test_60)

Información para horizonte de 60 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	193487 targets.
	Distribución y: mean=0.000114, std=0.003907, min=-0.040179, max=0.036224

Set de validación:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	41567 targets.
	Distribución y: mean=0.000163, std=0.004643, min=-0.038084, max=0.079896

Set de testeo:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	41567 targets.
	Distribución y: mean=-0.000068, std=0.003521, min=-0.018494, max=0.020365


### 2.3 Carga de ventanas 90 minutos

In [20]:
X_train_90_scaled, y_train_90, X_valid_90_scaled, y_valid_90, X_test_90_scaled, y_test_90, scaler_90 = load_windows_and_scaler(target = '90')

In [21]:
n_samples_train_90, n_samples_valid_90, n_samples_test_90 = xy_info('90', X_train_90_scaled, y_train_90, X_valid_90_scaled, y_valid_90, X_test_90_scaled, y_test_90)

Información para horizonte de 90 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	193487 targets.
	Distribución y: mean=0.000159, std=0.004746, min=-0.037742, max=0.039284

Set de validación:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	41567 targets.
	Distribución y: mean=0.000224, std=0.005887, min=-0.040748, max=0.083184

Set de testeo:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	41567 targets.
	Distribución y: mean=-0.000132, std=0.004558, min=-0.023392, max=0.026213


## 3. Dataset de Métricas

Dado que cada entrenamiento demanda un tiempo considerable, antes de proceder verificaremos si ya existe un resultado previo de este modelo consultando el dataset de métricas.

In [76]:
def load_metrics(data: str):
    data_path = f'{drive_path}/4_model_training/{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)
    return df

In [77]:
def metrics_verify(data: str) -> bool:
    data_path = f"{drive_path}/4_model_training/{data}.parquet"
    return os.path.exists(data_path)


In [131]:
if metrics_verify('4_1_random_forest_metrics'):
    print("Las métricas existen y son almacenadas random_forest_metrics")
    random_forest_metrics = load_metrics('4_1_random_forest_metrics')
    #print(random_forest_metrics)
    metrics = True
else:
    print("Las métricas no existen.")
    #Creamos la tabla para almacenar las métricas
    random_forest_metrics = pd.DataFrame(columns=["RMSE", "MAE", "R2", "SMAPE", "DirAcc"])
    metrics = False

Las métricas existen y son almacenadas random_forest_metrics


In [132]:
random_forest_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
RF_30_subsampleado_30%,0.002768,0.001628,0.237195,120.874522,0.698236
RF_30_subsampleado_50%,0.002782,0.001640,0.233573,121.386628,0.696868
RF_60_subsampleado_30%,0.003737,0.002029,0.326394,105.691331,0.760706
RF_60_subsampleado_50%,0.003946,0.002072,0.300929,105.248839,0.763990
RF_90_subsampleado_30%,0.004722,0.002363,0.352733,96.445722,0.794066
RF_90_subsampleado_50%,0.004718,0.002363,0.367801,95.253214,0.803349


## 4. Entrenamiento de Modelo

### 4.0. Funciones

#### Función para evaluación de modelo


In [133]:
# --- función de evaluación ---
def evaluate_model(model, X, y_true, y_pred, eps=1e-8):

    #Métricas básicas
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    # SMAPE (Symmetric Mean Absolute Percentage Error)
    smape = 100 * np.mean(
        (np.abs(y_true - y_pred) / ((np.abs(y_true) + np.abs(y_pred)) / 2 + eps))
    )

    # Directional Accuracy (acierto en el signo del retorno)
    directional_acc = np.mean(np.sign(y_true) == np.sign(y_pred))

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "SMAPE": smape,
        "DirAcc": directional_acc
    }

In [134]:
def print_metrics(metrics, target:str):
  print(f"Métricas de {target}:\n")
  for k, v in metrics.items():
      print(f"\t{k:>5}:\t {float(v):.6f}")

#### Función para subsamplear

In [135]:
def subsample(X, y, n):
    n = min(n, X.shape[0])
    idx = np.random.choice(X.shape[0], size=n, replace=False)
    return X[idx], y[idx]

#### Función para entrenamiento

In [136]:
def train_model (best_params, X_train, y_train, X_valid, y_valid):

    # Entrenamos el model final con los mejores hiperparámetros
    # Se entrena un RandomForest definitivo usando todos los datos de entrenamiento y los best_params
    best_model = RandomForestRegressor(**best_params, n_jobs=-1, random_state=42)
    best_model.fit(X_train, y_train)

    # Evaluación en validación
    preds = best_model.predict(X_valid)

    return best_model, preds

#### Parámetros por defecto

In [137]:
rf_default_params = {
    "n_estimators": 300,
    "max_depth": 15,
    "min_samples_split": 5,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "bootstrap": True,
    #"n_jobs": -1,
    #"random_state": 42
}

### 4.1. Entrenamiento 30min

#### 4.1.1. Con 30% de dataset

In [138]:
if metrics == False:
  print('Entrenando modelo RF_30_subsampleado_30%..')
  X_train_30_sub_1, y_train_30_sub_1 = subsample(X_train_30_scaled, y_train_30,  int(n_samples_train_30 * 0.3))
  X_valid_30_sub_1, y_valid_30_sub_1 = subsample(X_valid_30_scaled, y_valid_30,  int(n_samples_valid_30 * 0.3))
  model_30_sub_1, preds_30_sub_1 = train_model(rf_default_params, X_train_30_sub_1, y_train_30_sub_1, X_valid_30_sub_1, y_valid_30_sub_1)
  metrics_30_sub_1 = evaluate_model(model_30_sub_1, X_valid_30_sub_1, y_valid_30_sub_1, preds_30_sub_1)
  random_forest_metrics.loc["RF_30_subsampleado_30%"] = metrics_30_sub_1
elif metrics == True:
    print("El modelo fue entrenado anteriormente y las métricas ya fueron calculadas\n")
    metrics_30_sub_1 = random_forest_metrics.loc["RF_30_subsampleado_30%"]
    print_metrics(metrics_30_sub_1, '30 min subsampleado al 30%')

El modelo fue entrenado anteriormente y las métricas ya fueron calculadas

Métricas de 30 min subsampleado al 30%:

	 RMSE:	 0.002768
	  MAE:	 0.001628
	   R2:	 0.237195
	SMAPE:	 120.874522
	DirAcc:	 0.698236


#### 4.1.2. Con 50% de dataset

In [139]:
if metrics == False:
  print('Entrenando modelo RF_30_subsampleado_50%..')
  X_train_30_sub_2, y_train_30_sub_2 = subsample(X_train_30_scaled, y_train_30,int(n_samples_train_30 * 0.5))
  X_valid_30_sub_2, y_valid_30_sub_2 = subsample(X_valid_30_scaled, y_valid_30, int(n_samples_valid_30 * 0.5))
  model_30_sub_2, preds_30_sub_2 = train_model(rf_default_params, X_train_30_sub_2, y_train_30_sub_2, X_valid_30_sub_2, y_valid_30_sub_2)
  metrics_30_sub_2 = evaluate_model(model_30_sub_2, X_valid_30_sub_2, y_valid_30_sub_2, preds_30_sub_2)
  print_metrics(metrics_30_sub_2, 'Métricas de 30 min subsampleado al 50%')
  random_forest_metrics.loc["RF_30_subsampleado_50%"] = metrics_30_sub_2
elif metrics == True:
  print("El modelo fue entrenado anteriormente y las métricas ya fueron calculadas\n")
  metrics_30_sub_2 = random_forest_metrics.loc["RF_30_subsampleado_50%"]
  print_metrics(metrics_30_sub_2, '30 min subsampleado al 50%')

El modelo fue entrenado anteriormente y las métricas ya fueron calculadas

Métricas de 30 min subsampleado al 50%:

	 RMSE:	 0.002782
	  MAE:	 0.001640
	   R2:	 0.233573
	SMAPE:	 121.386628
	DirAcc:	 0.696868


#### 4.1.3. Con ventanas completas

In [ ]:
#model_30, preds_30 = train_model(rf_default_params, X_train_30_scaled, y_train_30, X_valid_30_scaled, y_valid_30)

In [ ]:
#metrics_30 = evaluate_model(model_30, X_valid_30_scaled, y_valid_30, preds_30)

In [ ]:
#print_metrics(metrics_30, '30 min completo')

In [ ]:
#random_forest_metrics.loc["RF_30_completo"] = metrics_30

### 4.2. Entrenamiento 60min

#### 4.2.1. Con 30% de dataset

In [122]:
if metrics == False:
  print('Entrenando modelo RF_60_subsampleado_30%..')
  X_train_60_sub_1, y_train_60_sub_1 = subsample(X_train_60_scaled, y_train_60,  int(n_samples_train_60 * 0.3))
  X_valid_60_sub_1, y_valid_60_sub_1 = subsample(X_valid_60_scaled, y_valid_60,  int(n_samples_valid_60 * 0.3))
  model_60_sub_1, preds_60_sub_1= train_model(rf_default_params, X_train_60_sub_1, y_train_60_sub_1, X_valid_60_sub_1, y_valid_60_sub_1)
  metrics_60_sub_1 = evaluate_model(model_60_sub_1, X_valid_60_sub_1, y_valid_60_sub_1, preds_60_sub_1)
  print_metrics(metrics_60_sub_1, '60 min subsampleado al 30%' )
  random_forest_metrics.loc["RF_60_subsampleado_30%"] = metrics_60_sub_1
elif metrics == True:
  print("El modelo fue entrenado anteriormente y las métricas ya fueron calculadas\n")
  metrics_60_sub_1 = random_forest_metrics.loc["RF_60_subsampleado_30%"]
  print_metrics(metrics_60_sub_1, 'Métricas de 60 min subsampleado al 30%' )


El modelo fue entrenado anteriormente y las métricas ya fueron calculadas

Métricas de Métricas de 60 min subsampleado al 30%:

	 RMSE:	 0.003737
	  MAE:	 0.002029
	   R2:	 0.326394
	SMAPE:	 105.691331
	DirAcc:	 0.760706


#### 4.2.2. Con 50% de dataset

In [123]:
if metrics == False:
  print('Entrenando modelo RF_60_subsampleado_50%..')
  X_train_60_sub_2, y_train_60_sub_2 = subsample(X_train_60_scaled, y_train_60,int(n_samples_train_60 * 0.5))
  X_valid_60_sub_2, y_valid_60_sub_2 = subsample(X_valid_60_scaled, y_valid_60, int(n_samples_valid_60 * 0.5))
  model_60_sub_2, preds_60_sub_2 = train_model(rf_default_params, X_train_60_sub_2, y_train_60_sub_2, X_valid_60_sub_2, y_valid_60_sub_2)
  metrics_60_sub_2 = evaluate_model(model_60_sub_2, X_valid_60_sub_2, y_valid_60_sub_2, preds_60_sub_2)
  print_metrics(metrics_60_sub_2, '60 min subsampleado al 50%')
  random_forest_metrics.loc["RF_60_subsampleado_50%"] = metrics_60_sub_2
elif metrics == True:
  print("El modelo fue entrenado anteriormente y las métricas ya fueron calculadas\n")
  metrics_60_sub_2 = random_forest_metrics.loc["RF_60_subsampleado_50%"]
  print_metrics(metrics_60_sub_2, 'Métricas de 60 min subsampleado al 50%' )

El modelo fue entrenado anteriormente y las métricas ya fueron calculadas

Métricas de Métricas de 60 min subsampleado al 50%:

	 RMSE:	 0.003946
	  MAE:	 0.002072
	   R2:	 0.300929
	SMAPE:	 105.248839
	DirAcc:	 0.763990


#### 4.2.3. Con ventanas completas

In [ ]:
#model_60, preds_60 = train_model(rf_default_params, X_train_60_scaled, y_train_60, X_valid_60_scaled, y_valid_60)

In [ ]:
#metrics_60 = evaluate_model(model_60, X_valid_60_scaled, y_valid_60, preds_60)

In [ ]:
#print_metrics(metrics_60, '60 min completo')

In [ ]:
#random_forest_metrics.loc["RF_60_completo"] = metrics_60

### 4.3. Entrenamiento 90min

#### 4.3.1. Con 30% de dataset

In [125]:
if metrics == False:
  print('Entrenando modelo RF_90_subsampleado_30%..')
  X_train_90_sub_1, y_train_90_sub_1 = subsample(X_train_90_scaled, y_train_90,  int(n_samples_train_90 * 0.3))
  X_valid_90_sub_1, y_valid_90_sub_1 = subsample(X_valid_90_scaled, y_valid_90,  int(n_samples_valid_90 * 0.3))
  model_90_sub_1, preds_90_sub_1= train_model(rf_default_params, X_train_90_sub_1, y_train_90_sub_1, X_valid_90_sub_1, y_valid_90_sub_1)
  metrics_90_sub_1 = evaluate_model(model_90_sub_1, X_valid_90_sub_1, y_valid_90_sub_1, preds_90_sub_1)
  print_metrics(metrics_90_sub_1, '90 min subsampleado al 30%' )
  random_forest_metrics.loc["RF_90_subsampleado_30%"] = metrics_90_sub_1
elif metrics == True:
  print("El modelo fue entrenado anteriormente y las métricas ya fueron calculadas\n")
  metrics_90_sub_1 = random_forest_metrics.loc["RF_90_subsampleado_30%"]
  print_metrics(metrics_90_sub_1, 'Métricas de 90 min subsampleado al 30%' )

El modelo fue entrenado anteriormente y las métricas ya fueron calculadas

Métricas de Métricas de 90 min subsampleado al 30%:

	 RMSE:	 0.004722
	  MAE:	 0.002363
	   R2:	 0.352733
	SMAPE:	 96.445722
	DirAcc:	 0.794066


#### 4.3.2. Con 50% de dataset

In [126]:
if metrics == False:
  print('Entrenando modelo RF_90_subsampleado_50%..')
  X_train_90_sub_2, y_train_90_sub_2 = subsample(X_train_90_scaled, y_train_90,int(n_samples_train_90 * 0.5))
  X_valid_90_sub_2, y_valid_90_sub_2 = subsample(X_valid_90_scaled, y_valid_90, int(n_samples_valid_90 * 0.5))
  model_90_sub_2, preds_90_sub_2 = train_model(rf_default_params, X_train_90_sub_2, y_train_90_sub_2, X_valid_90_sub_2, y_valid_90_sub_2)
  metrics_90_sub_2 = evaluate_model(model_90_sub_2, X_valid_90_sub_2, y_valid_90_sub_2, preds_90_sub_2)
  print_metrics(metrics_90_sub_2, '90 min subsampleado al 50%')
  random_forest_metrics.loc["RF_90_subsampleado_50%"] = metrics_90_sub_2
elif metrics == True:
  print("El modelo fue entrenado anteriormente y las métricas ya fueron calculadas\n")
  metrics_90_sub_2 = random_forest_metrics.loc["RF_90_subsampleado_50%"]
  print_metrics(metrics_90_sub_2, 'Métricas de 90 min subsampleado al 50%' )

El modelo fue entrenado anteriormente y las métricas ya fueron calculadas

Métricas de Métricas de 90 min subsampleado al 50%:

	 RMSE:	 0.004718
	  MAE:	 0.002363
	   R2:	 0.367801
	SMAPE:	 95.253214
	DirAcc:	 0.803349


#### 4.3.3. Con ventanas completas

In [ ]:
#model_90, preds_90 = train_model(rf_default_params, X_train_90_scaled, y_train_90, X_valid_90_scaled, y_valid_90)

In [ ]:
#metrics_90 = evaluate_model(model_90, X_valid_90_scaled, y_valid_90, preds_90)

In [ ]:
#print_metrics(metrics_90, '90 min completo')

In [ ]:
#random_forest_metrics.loc["RF_90_completo"] = metrics_90

## 5. Recuperación de métricas

Este punto existe para garantizar reproducibilidad y continuidad del análisis sin reentrenar modelos cuando se pierden las métricas. Actúa como fallback: reconstruye la tabla de métricas de Random Forest a partir de valores ya validados y la persiste nuevamente, evitando el re-entrenamiento de los modelos.

Así, se mantiene la consistencia de resultados y la trazabilidad de comparaciones y conclusiones, incluso si el archivo original fue eliminado, corrompido o el entorno de ejecución cambió.

In [127]:
def generate_metrics_RF():
  '''
  Ejecutar está función solo en caso de perder las métricas
  El objetivo es no volver a correr los entrenamientos
  '''
  random_forest_metrics = pd.DataFrame(columns=["RMSE", "MAE", "R2", "SMAPE", "DirAcc"])

  metrics_30_sub_1 = {
      "RMSE": 0.002768,
      "MAE": 0.001628,
      "R2": 0.237195,
      "SMAPE": 120.874522,
      "DirAcc": 0.698236
  }


  metrics_30_sub_2 = {
    "RMSE":  0.002782,
    "MAE":   0.001640,
    "R2":  0.233573,
    "SMAPE":   121.386628,
    "DirAcc": 0.696868,
  }

  metrics_60_sub_1 = {
      "RMSE": 0.003737,
      "MAE": 0.002029,
      "R2": 0.326394,
      "SMAPE": 105.691331,
      "DirAcc": 0.760706
  }


  metrics_60_sub_2 = {
      "RMSE": 0.003946,
      "MAE": 0.002072,
      "R2": 0.300929,
      "SMAPE": 105.248839,
      "DirAcc": 0.763990
  }

  metrics_90_sub_1 = {
      "RMSE": 0.004722,
      "MAE": 0.002363,
      "R2": 0.352733,
      "SMAPE": 96.445722,
      "DirAcc": 0.794066
  }

  metrics_90_sub_2 = {
      "RMSE": 0.004718,
      "MAE": 0.002363,
      "R2": 0.367801,
      "SMAPE": 95.253214,
      "DirAcc": 0.803349
  }


  random_forest_metrics.loc["RF_30_subsampleado_30%"] = metrics_30_sub_1
  random_forest_metrics.loc["RF_30_subsampleado_50%"] = metrics_30_sub_2
  random_forest_metrics.loc["RF_60_subsampleado_30%"] = metrics_60_sub_1
  random_forest_metrics.loc["RF_60_subsampleado_50%"] = metrics_60_sub_2
  random_forest_metrics.loc["RF_90_subsampleado_30%"] = metrics_90_sub_1
  random_forest_metrics.loc["RF_90_subsampleado_50%"] = metrics_90_sub_2


  random_forest_metrics_path = f'{drive_path}/4_model_training/4_1_random_forest_metrics.parquet'
  random_forest_metrics.to_parquet(random_forest_metrics_path, index=True)


In [128]:
#Ejecutar esta función solo en caso de perder las métricas de entrenamiento
#generate_metrics_RF()